# 05 · Social Recommendation

Use the **friend graph** to recommend: a friend's tastes inform yours, most useful when your own history is thin.

**Social-neighbor CF:**

$$score(u, i) = (1-\alpha)\cdot \underbrace{\sum_{f \in friends(u)} trust(u,f)\cdot liked(f, i)}_{\text{social}} + \alpha \cdot popularity(i)$$

The popularity term is a small back-off so friendless/cold users still get sensible recs.

> **Honesty caveat:** on the **synthetic** dataset the friend graph is partly built from latent taste, so results here validate the *mechanics* only. The real scientific test ("does social help?") needs the **real Yelp friend graph** — same code, just point `load_dataset()` at the Yelp subset.

In [ ]:
import os
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

import pandas as pd

from src.recsys.config import settings
from src.recsys.data import load_dataset
from src.recsys.eval import evaluate, evaluate_by_activity, temporal_split
from src.recsys.models import ALSRecommender, PopularityRecommender, SocialRecommender

ds = load_dataset()
train, test_positives = temporal_split(ds.interactions)
test_users = list(test_positives.keys())
max_k = max(settings.eval_ks)
train_counts = train.groupby(settings.user_col).size().to_dict()
print(ds.summary())
print("social edges:", ds.n_social_edges)

## Social vs baselines

Two trust variants (uniform vs Jaccard overlap) compared against popularity and ALS on the same split.

In [ ]:
models = {
    "popularity": PopularityRecommender(),
    "social_uniform": SocialRecommender(social=ds.social, alpha=0.15, trust="uniform"),
    "social_jaccard": SocialRecommender(social=ds.social, alpha=0.15, trust="jaccard"),
    "als": ALSRecommender(factors=64, iterations=15),
}

results, all_recs = {}, {}
for name, model in models.items():
    model.fit(train)
    recs = model.recommend(test_users, k=max_k)
    all_recs[name] = recs
    results[name] = evaluate(recs, test_positives, ks=settings.eval_ks, n_items=ds.n_items)

results_df = pd.DataFrame(results).T
results_df[[c for c in results_df.columns if c.endswith(f"@{settings.top_k}")]]

## The key view: sliced by user activity

Social signal should help **sparse** users most. This slices metrics by how much training history each user had.

In [ ]:
rows = []
for name in ["social_jaccard", "als"]:
    sliced = evaluate_by_activity(all_recs[name], test_positives, train_counts, k=settings.top_k)
    for bucket, m in sliced.items():
        rows.append({
            "model": name, "activity": bucket, "n_users": m.get("n_users", 0),
            f"recall@{settings.top_k}": m.get(f"recall@{settings.top_k}", float("nan")),
            f"ndcg@{settings.top_k}": m.get(f"ndcg@{settings.top_k}", float("nan")),
        })
pd.DataFrame(rows)

## How to read this

- **Social >> popularity** but **< ALS** is the typical pattern for pure social-neighbor CF: friends are informative, but a well-tuned collaborative model using *all* co-occurrence is stronger. That's an honest, useful finding — not a failure.
- The real value of social usually shows up **for sparse users** and when **combined** with a collaborative model (social as a *feature*, e.g. into the two-stage ranker or as social regularization on MF), not as a standalone recommender.
- On **synthetic** data the friend graph is taste-derived, so treat these numbers as mechanics validation. Re-run on the **real Yelp subset** for the honest lift test — same code.

**Next steps to try:**
- Feed a `social_score` feature into the `LightGBMRanker` (two-stage) and measure lift.
- Social-regularized matrix factorization (SoReg / SocialMF).
- Then Yelp data → honest evaluation, sliced by activity (cold users should benefit most).